# 第4周：性能与排障

> **学习目标**：掌握 CPU/内存/磁盘 I/O 性能分析工具，学会使用 strace 调试系统调用、lsof 排查一切文件问题、cgroup v2 资源限制、综合故障排查方法论

---


## 开篇：故障排查工程师的武器库

前三周我们学习了用户空间、系统管理和网络。这周我们把所有知识串起来，学习**性能分析和故障排查**。

这周结束时，你将成为团队里的"救火队员"——遇到线上问题不再慌，而是有条不紊地用工具链定位根因。

- CPU 100% 了，谁干的？ -> top -> perf top -> 火焰图
- 内存不够了，OOM 杀了谁？ -> dmesg -> /proc/<pid>/smaps
- 磁盘很慢，瓶颈在哪？ -> iostat -> iotop -> fio
- 程序起不来，为什么？ -> strace 看到一切系统调用
- 磁盘满了但 du 找不到大文件？ -> lsof | grep deleted
- 怎么限制容器资源？ -> cgroup v2

最重要的是**方法论**：现象 -> 排查 -> 根因 -> 修复。

---
## Day 22：CPU 性能分析

### 工具速览

| 工具 | 用途 | 安装 |
|------|------|------|
| top / htop | 实时进程监控 | apt install htop |
| mpstat | 每个 CPU 核心的使用率 | apt install sysstat |
| pidstat | 按进程的 CPU 统计 | apt install sysstat |
| perf | 性能事件采样（函数级） | apt install linux-tools-common |

### top -- 排查 CPU 问题的第一站

当你收到告警说服务器 CPU 高时，第一件事就是登录上去运行 `top`。

看什么：
- `load average`：1/5/15 分钟的平均负载。超过 CPU 核心数说明有瓶颈
- `us`：用户空间 CPU 时间（你的代码）
- `sy`：系统调用时间（内核）
- `wa`：等待 I/O 的时间（磁盘慢时这个值会高）
- `hi`：硬件中断；`si`：软件中断
- `st`：被虚拟机偷走的时间（云服务器被超卖时）

### mpstat -P ALL 1

显示每个 CPU 核心的独立统计。如果某个核的 us 远高于其他核，说明有单线程程序卡在某个核上。
如果所有核都很忙，那就是多线程/多进程的工作负载。

### pidstat -p <PID> 1

定向追踪某个进程的 CPU 使用。配合 `-t` 可以看到线程级别的统计。
`pidstat 1` 不带 PID 会显示所有活跃进程。

### perf -- 函数级性能分析

top 和 pidstat 只能看到进程级别，perf 能深入到**函数级别**：

```bash
perf top -p <PID>                # 实时显示最热的函数
perf record -p <PID> -g sleep 30 # 采样 30 秒，记录调用栈
perf report                      # 分析采样结果
```

如果某个函数占用了 80% 的 CPU，那这就是你要优化的目标。

### 火焰图

从 perf 数据生成火焰图，直观展示 CPU 时间都花在哪了。
每个矩形代表一个函数调用，宽度越宽说明占用的 CPU 时间越多。
找最宽的矩形，看它的调用路径——这就是瓶颈所在。

```bash
perf record -a -g -- sleep 30
perf script > out.perf
# 用 Brendan Gregg 的 FlameGraph 工具生成 SVG
```


In [ ]:
# CPU 性能分析实验
! echo "=== 制造 CPU 负载 ==="
! python3 -c "import hashlib; [hashlib.sha256(b'x'*50000).hexdigest() for _ in range(100000)]" &
! CPUPID=$!
! echo "CPU 密集型进程 PID: $CPUPID"
! sleep 1
echo ""
! echo "=== top 查看（批处理模式） ==="
! top -b -n1 -p $CPUPID 2>/dev/null | tail -3 || echo "top 不可用"
echo ""
! echo "=== CPU 信息 ==="
! echo "CPU 核心数: $(nproc)"
! grep 'model name' /proc/cpuinfo | head -1
! echo ""
! echo "=== load average 解读 ==="
! LOAD=$(uptime | awk -F'load average:' '{print $2}')
! echo "当前负载: $LOAD"
! echo "CPU 核心数: $(nproc)"
! echo "负载/核心数 = 需要排查？"
! kill $CPUPID 2>/dev/null || true
echo ""
! echo "=== 谁占 CPU 最多 ==="
! ps aux --sort=-%cpu 2>/dev/null | awk 'NR<=4 {printf "%-6s %-25s %s%%\n", $2, $11, $3}' || echo "无权限"
echo ""
! echo "=== 查看系统调用占比 ==="
! cat /proc/stat | head -1
! echo "（us=用户, sy=系统, ni=优先级, id=空闲, wa=I/O, hi=硬中断, si=软中断, st=被偷走）"

---
## Day 23：内存分析

### VIRT / RES / SHR -- 进程内存三要素

| 指标 | 全称 | 含义 | 排障价值 |
|------|------|------|----------|
| VIRT | Virtual Memory | 进程申请的虚拟内存总量 | 一般没用，很多是 mmap 的共享库 |
| RES | Resident Memory | 实际占用的物理内存 | **最有价值**，看它是否持续增长 |
| SHR | Shared Memory | 共享内存（如共享库） | 多进程间共享，不计入实际消耗 |

VIRT 高不代表内存泄漏！真正要看的是 RES。
很多程序预先申请大量虚拟内存（如 JVM 默认堆），但实际使用的物理内存并不多。

### OOM Killer

当系统内存不足时，内核启动 OOM Killer 选择一个进程杀掉。选择标准：
1. 占用内存最多的进程
2. 优先级低的进程（oom_score_adj 调低可以降低被杀概率）

查看 OOM 日志：

```bash
dmesg | grep -i oom              # 内核日志中的 OOM
dmesg | grep -i "killed process" # 看谁被杀死了
```

查看进程的 OOM 评分：

```bash
cat /proc/<pid>/oom_score        # 分数越高越容易被杀
cat /proc/<pid>/oom_score_adj    # -1000 到 1000，可以手动调整
```

### /proc/<pid>/smaps -- 内存映射细节

每一段映射包含：地址范围、权限、偏移、设备、inode、路径名、RSS/PSS 统计。
PSS（Proportional Set Size）比 RSS 更准确，因为它按比例分摊了共享内存。

### pmap -x <pid>

快速查看进程内存映射快照。比 smaps 更易读。

### 内存泄漏排查步骤

1. 用 `ps aux --sort=-%mem` 观察 RES 是否持续增长
2. 每隔 30 秒执行一次，看趋势
3. 用 `valgrind --tool=memcheck ./program` 检查泄漏
4. 或用 `heaptrack`（更现代的工具，输出可视化的内存分析报告）


In [ ]:
# 内存分析实验
! echo "=== 进程内存三要素 ==="
! ps -p $$ -o pid,comm,vsz,rss,sz | head -5
echo ""
! echo "=== 按内存使用量排序 ==="
! ps aux --sort=-%mem 2>/dev/null | awk 'NR<=5 {printf "%-6s %-25s %s\n", $2, $11, $4"%"}' || echo "无权限"
echo ""
! echo "=== 当前进程内存详情 ==="
! grep -E "VmRSS|VmSize|VmPeak|VmSwap" /proc/$$/status
echo ""
! echo "=== OOM 评分 ==="
! if [ -f /proc/$$/oom_score ]; then
!     echo "当前进程 oom_score: $(cat /proc/$$/oom_score)"
!     echo "oom_score_adj: $(cat /proc/$$/oom_score_adj 2>/dev/null)"
! fi
echo ""
! echo "=== 系统内存总览 ==="
! free -h
! echo ""
! echo "=== /proc/meminfo 关键字段 ==="
! grep -E "MemTotal|MemFree|MemAvailable|Buffers|Cached" /proc/meminfo
echo ""
! echo "=== OOM 排查命令速记 ==="
! echo "dmesg | grep -i oom"
! echo "dmesg | grep -i 'killed process'"
! echo "cat /proc/<pid>/oom_score"
! echo "echo -1000 > /proc/<pid>/oom_score_adj  # 降低被杀概率"

---
## Day 24：磁盘 I/O 分析

### iostat -x 1 -- 磁盘 I/O 统计

最常用的磁盘性能分析工具：

| 字段 | 含义 | 正常范围 |
|------|------|----------|
| r/s | 每秒读请求数 | 取决于磁盘类型 |
| w/s | 每秒写请求数 | 取决于磁盘类型 |
| rKB/s | 每秒读数据量 | |
| wKB/s | 每秒写数据量 | |
| await | I/O 平均等待时间 (ms) | <10ms 优秀, >50ms 有问题 |
| svctm | 服务时间 (ms) | 磁盘本身处理时间 |
| %util | 磁盘忙碌百分比 | >80% 说明磁盘是瓶颈 |

> await 很高但 svctm 正常 = 请求在队列里等太久（磁盘忙不过来）
> await 和 svctm 都高 = 磁盘本身慢（可能是硬件问题）

### iotop -o -- 按进程看 I/O

iostat 只能看到磁盘整体情况，iotop 能看到哪个进程在疯狂读写：
`sudo iotop -o` 只显示有 I/O 的进程，`sudo iotop -b -n 5` 批处理 5 次。

### fio -- 基准测试

当你想知道磁盘的极限性能时，用 fio：

```bash
fio --name=test --rw=randrw --bs=4k --size=1G --numjobs=4 --runtime=30
```

参数：--rw=randrw 随机读写, --bs=4k 块大小, --numjobs=4 并发数。

### du / ncdu -- 磁盘空间分析

```bash
du -sh /var/log           # 查看目录总大小
du -sh * | sort -rh       # 当前目录按大小排序
ncdu /var                 # 交互式分析（需要安装）
```

**注意**：du 遍历所有文件，大目录可能很慢。先 cd 到目录再运行更准确。


In [ ]:
# 磁盘 I/O 分析实验
! echo "=== 磁盘信息 ==="
! lsblk 2>/dev/null
echo ""
! echo "=== 文件系统使用 ==="
! df -h 2>/dev/null
echo ""
! echo "=== iostat（如果已安装） ==="
! if command -v iostat &>/dev/null; then
!     iostat -x 1 1 2>/dev/null | tail -8 || echo "需要更高权限"
! else
!     echo "iostat 未安装。执行: sudo apt install sysstat"
! fi
echo ""
! echo "=== dd 写入/读取性能测试 ==="
! echo "--- 写入 256MB ---"
! dd if=/dev/zero of=/tmp/iotest bs=1M count=256 2>&1 | tail -1
! echo "--- 读取 256MB ---"
! dd if=/tmp/iotest of=/dev/null bs=1M 2>&1 | tail -1
! rm -f /tmp/iotest
echo ""
! echo "=== du 使用示例 ==="
! du -sh /tmp 2>/dev/null | awk '{print "临时目录大小: " $1}'
echo ""
! echo "=== /proc/diskstats ==="
! cat /proc/diskstats 2>/dev/null | head -8 || echo "无权限"
echo ""
! echo "=== fio 命令示例 ==="
! echo "fio --name=test --rw=randrw --bs=4k --size=1G --runtime=30"
! echo "（运行需要几秒到几分钟，取决于磁盘速度）"

---
## Day 25：strace -- 系统调用调试

### strace 的原理

strace 使用 ptrace 系统调用来拦截目标进程的每一次系统调用。
每次调用打印：调用名、参数、返回值。

```
write(1, "hello\n", 6)          = 6
open("/etc/passwd", O_RDONLY)   = 3
read(3, "root:x:0:0:root:", 1024) = 20
close(3)                         = 0
```

### 常用用法

| 用法 | 用途 |
|------|------|
| `strace -p <pid>` | 追踪正在运行的进程 |
| `strace -c <cmd>` | 运行命令并统计系统调用耗时 |
| `strace -e trace=file <cmd>` | 只追踪文件相关调用 |
| `strace -e trace=network <cmd>` | 只追踪网络相关调用 |
| `strace -e trace=process <cmd>` | 只追踪进程相关调用 |
| `strace -f <cmd>` | 追踪子进程 |
| `strace -T <cmd>` | 显示每次调用的耗时（微秒） |
| `strace -o out.log <cmd>` | 输出到文件 |

### 三个经典排查场景

**场景 1：程序启动失败**
配置文件找不到时，strace 会显示：
`open("/etc/myapp.conf", O_RDONLY) = -1 ENOENT (No such file or directory)`
```bash
strace -e trace=file myapp 2>&1 | grep ENOENT
```

**场景 2：权限不够**
没有权限时，strace 会显示：
`open("/var/log/myapp.log", O_WRONLY) = -1 EACCES (Permission denied)`
```bash
strace -e trace=file myapp 2>&1 | grep EACCES
```

**场景 3：程序到底在忙什么**
```bash
strace -c -p <pid>         # 统计系统调用分布
strace -T -p <pid>         # 看哪个调用最耗时
```
如果 poll/select 调用占了 99%，说明程序在等待 I/O。
如果 read/write 占了 99%，说明程序在大量读写数据。


In [ ]:
# strace 实验
! echo "=== strace 追踪简单命令 ==="
! if command -v strace &>/dev/null; then
!     strace echo hello 2>&1 | head -10
! else
!     echo "strace 未安装。执行: sudo apt install strace"
! fi
echo ""
! echo "=== 系统调用次数统计 ==="
! if command -v strace &>/dev/null; then
!     strace -c echo test 2>&1 | tail -16
! fi
echo ""
! echo "=== 只追踪文件操作 ==="
! if command -v strace &>/dev/null; then
!     strace -e trace=file cat /etc/hostname 2>&1 | grep -E "open|ENOENT|EACCES" | head -5
! fi
echo ""
! echo "=== 追踪网络调用 ==="
! if command -v strace &>/dev/null && command -v curl &>/dev/null; then
!     strace -e trace=network curl -s http://example.com 2>&1 | grep -E "connect|socket" | head -5
! fi
echo ""
! echo "=== 查看系统调用耗时 ==="
! if command -v strace &>/dev/null; then
!     strace -T echo test 2>&1 | grep "<" | head -3
! fi
echo ""
! echo "=== strace 实用技巧 ==="
! echo "1. strace -f docker run --rm alpine echo hi  # 看 Docker 启动流程"
! echo "2. strace -e trace=file nginx 2>&1 | grep conf  # 看 nginx 读哪个配置文件"
! echo "3. strace -p $(pgrep -f sshd) -e trace=network  # 看 sshd 的网络调用"

---
## Day 26：lsof -- 一切皆文件

Linux 的设计哲学是**一切皆文件**——普通文件、目录、socket、管道、设备……都是文件。
`lsof` 可以列出所有打开的文件，是排查问题的瑞士军刀。

### 关键用法速查

| 命令 | 场景 |
|------|------|
| `lsof /var/log/syslog` | 哪个进程在写这个日志文件？ |
| `lsof -p <pid>` | 这个进程打开了哪些文件？ |
| `lsof -i :80` | 哪个进程在监听 80 端口？ |
| `lsof -u <user>` | 某个用户打开了哪些文件？ |
| `lsof | grep deleted` | 已删除但还被进程占用的文件 |
| `lsof -p <pid> | grep cwd` | 进程的工作目录在哪？ |
| `lsof -i :3000-4000` | 3000-4000 端口范围的监听 |
| `lsof -iTCP -sTCP:LISTEN` | 所有 TCP 监听 socket |

### 经典场景：磁盘空间消失

这是 Linux 面试必考题，也是真实世界天天发生的问题：

```bash
# 1. df 显示磁盘满了
df -h /var

# 2. du 找不到大文件
du -sh /var/* | sort -rh | head -5

# 3. 真相：有进程打开了已删除的文件
lsof | grep deleted

# 4. 修复：
sudo truncate -s 0 /proc/<pid>/fd/<n>   # 清空文件内容
# 或: kill -HUP <pid>                    # 让进程重开日志文件
```

**原理**：rm 删除了目录条目，但文件的 inode 引用计数因为进程还持有 fd 而没有归零。
内核要等到最后一个引用关闭后才释放磁盘空间。

### 端口冲突排查

启动服务时提示"Address already in use"：
```bash
lsof -i :8080                 # 谁占了 8080？
kill -9 <pid>                 # 杀掉它（确认没问题后）
# 或换一个端口
```


In [ ]:
# lsof 实验
! echo "=== 当前 Shell 打开的文件 ==="
! if command -v lsof &>/dev/null; then
!     lsof -p $$ 2>/dev/null | head -10
! else
!     echo "lsof 未安装。执行: sudo apt install lsof"
! fi
echo ""
! echo "=== 查看端口 22 的监听进程 ==="
! if command -v lsof &>/dev/null; then
!     lsof -i :22 2>/dev/null | head -5
! fi
echo ""
! echo "=== 模拟磁盘空间消失经典问题 ==="
! exec 3> /tmp/.big_hidden_file
! dd if=/dev/zero of=/tmp/.big_hidden_file bs=1M count=30 2>/dev/null
! echo "写入 30MB 后执行 rm ..."
! rm -f /tmp/.big_hidden_file
! echo "文件已删除！但 fd 3 还开着"
! echo ""
! echo "查找 deleted 文件："
! lsof 2>/dev/null | grep deleted | head -3 || echo "（lsof 受限或无 deleted 文件）"
! echo ""
! echo "修复：关闭 fd"
! exec 3>&-
! echo "fd 3 已关闭，空间释放"
echo ""
! echo "=== lsof 输出格式 ==="
! echo "FD 列含义："
! echo "  cwd = 当前工作目录"
! echo "  rtd = 根目录"
! echo "  txt = 程序代码段"
! echo "  0u = fd 0 (stdin)"
! echo "  1u = fd 1 (stdout)"
! echo "  2u = fd 2 (stderr)"
! echo "  3w = fd 3 (只写)"

In [ ]:
# cgroup v2 实验
! echo "=== 检查 cgroup 版本 ==="
! if [ -f /sys/fs/cgroup/cgroup.controllers ]; then
!     echo "系统使用 cgroup v2"
!     echo "活跃控制器: $(cat /sys/fs/cgroup/cgroup.controllers)"
! else
!     echo "系统使用 cgroup v1（较旧）"
! fi
echo ""
! echo "=== 当前进程的 cgroup ==="
! cat /proc/self/cgroup 2>/dev/null || echo "无法读取"
echo ""
! echo "=== 查看系统级资源限制 ==="
! if [ -d /sys/fs/cgroup ]; then
!     echo "--- CPU 限制 ---"
!     cat /sys/fs/cgroup/cpu.max 2>/dev/null || echo "（不可读）"
!     echo "--- 内存限制 ---"
!     cat /sys/fs/cgroup/memory.max 2>/dev/null || echo "（不可读）"
!     echo "--- 当前内存 ---"
!     cat /sys/fs/cgroup/memory.current 2>/dev/null | awk '{print $1/1024/1024 " MB"}' || echo "（不可读）"
!     echo "--- 进程数限制 ---"
!     cat /sys/fs/cgroup/pids.max 2>/dev/null || echo "（不可读）"
! fi
echo ""
! echo "=== 创建测试 cgroup（需要 root） ==="
! sudo mkdir -p /sys/fs/cgroup/perf_test 2>/dev/null && echo "cgroup 创建成功" || echo "创建失败（需要 root 权限，这在容器中很常见）"
! if [ -d /sys/fs/cgroup/perf_test ]; then
!     echo ""
!     echo "设置 50MB 内存限制:"
!     echo "52428800" | sudo tee /sys/fs/cgroup/perf_test/memory.max
!     echo "设置 0.5 核 CPU 限制:"
!     echo "50000 100000" | sudo tee /sys/fs/cgroup/perf_test/cpu.max
!     echo "设置 10 个进程限制:"
!     echo "10" | sudo tee /sys/fs/cgroup/perf_test/pids.max
!     echo ""
!     echo "验证配置："
!     echo "CPU: $(cat /sys/fs/cgroup/perf_test/cpu.max)"
!     echo "Mem: $(cat /sys/fs/cgroup/perf_test/memory.max) (bytes)"
!     echo "PID: $(cat /sys/fs/cgroup/perf_test/pids.max)"
!     sudo rmdir /sys/fs/cgroup/perf_test && echo "已清理"
! fi
echo ""
! echo "=== Docker 容器 cgroup 对应 ==="
! echo "docker run --memory=256m --cpus=1.5 --name test alpine sleep 3600"
! echo "这会在以下路径创建 cgroup："
! echo "  /sys/fs/cgroup/system.slice/docker-<id>.scope/"
! echo "其中 memory.max = 268435456, cpu.max = 150000 100000"

---
## Day 28：第四周综合练习 -- 故障排查演练

### 6 个真实故障场景

每个场景的目标：理解现象 -> 确定排查工具链 -> 找到根因 -> 提出修复方案 -> 记录笔记。

---
### 场景 1：服务起不来

**现象**：`systemctl start myapp` 返回成功，但进程几秒后就挂了。

**排查工具链**：
1. `systemctl status myapp.service` -- 看退出码和错误信息
2. `journalctl -u myapp.service -n 50` -- 看日志
3. `ss -tlnp | grep <port>` -- 端口被占用？
4. `strace -f -o /tmp/trace.log myapp` -- 看系统调用
5. `grep -E "ENOENT|EACCES" /tmp/trace.log` -- 找文件错误

**常见根因**：端口被占 / 配置文件不存在 / 权限不够 / 环境变量缺失

---
### 场景 2：磁盘空间告急

**现象**：df -h 显示使用率 100%，但 du -sh 加起来只有一半。

**排查工具链**：
1. `df -h` -- 确认磁盘满了
2. `du -sh * | sort -rh | head -10` -- 找不到大文件
3. `lsof | grep deleted` -- 发现被删除但还开着的文件
4. `sudo truncate -s 0 /proc/<pid>/fd/<n>` -- 清空文件释放空间

**根因**：进程写日志 -> rm 删除日志 -> 进程没重启 -> fd 还在 -> 空间不释放

---
### 场景 3：OOM Killer

**现象**：关键进程突然消失，进程列表找不到，系统日志无错误。

**排查工具链**：
1. `dmesg | grep -i oom` -- 确认是 OOM
2. `dmesg | grep -i 'killed process'` -- 看谁被杀了
3. `free -h` -- 确认内存不足
4. `ps aux --sort=-%mem | head -10` -- 谁在吃内存

**修复**：增加内存 / 调低关键进程的 oom_score_adj / 排查内存泄漏

---
### 场景 4：网络超时

**现象**：curl http://api.example.com 卡住很久然后超时。

**排查工具链**：
1. `dig api.example.com` -- DNS 解析正常吗？
2. `ping api.example.com` -- 网络通吗？
3. `ss -tan | grep :443` -- TCP 连接状态？
4. `traceroute api.example.com` -- 路由在哪跳卡住？
5. `iptables -L -n` -- 防火墙拦截？

---
### 场景 5：CPU 100%

**现象**：top 显示 CPU 100%，某个进程占满，但不知道在执行什么代码。

**排查工具链**：
1. `top -p <PID>` -- 确认 CPU 高
2. `pidstat -p <PID> -t 1` -- 看哪个线程
3. `perf top -p <PID>` -- 看最热的函数
4. `perf record -p <PID> -g sleep 30` -- 采样
5. `perf report` -- 分析或生成火焰图

**常见根因**：死循环 / 正则是回溯 / 频繁 GC / 内核驱动

---
### 场景 6：容器内慢

**现象**：同一应用在容器内比宿主机慢 10 倍，但宿主机资源充足。

**排查工具链**：
1. `docker stats <container>` -- 看资源限制
2. `cat /sys/fs/cgroup/system.slice/docker-<id>.scope/cpu.max` -- 发现 CPU 限制太低
3. `iostat -x 1` -- 磁盘 I/O 瓶颈
4. `strace -T -p <pid>` -- 看哪个系统调用慢

---
### 排障笔记模板

对每个场景，用以下格式记录：

```
## 场景名称
- 现象：
- 排查过程：
  1. xx 工具 -> 发现 yy
  2. xx 工具 -> 确认 zz
- 根因：
- 修复方案：
- 预防措施：
```


In [ ]:
# 综合故障排查演练
! echo "========================================"
! echo "  第四周综合练习：6 个故障场景"
! echo "========================================"
echo ""
! echo "=== 场景 1：服务起不来 ==="
! echo "模拟检查：请在实际环境中执行以下命令"
! echo "  systemctl status nginx"
! echo "  journalctl -u nginx -n 20"
! echo "  ss -tlnp | grep 80"
! echo "  nginx -t"
! systemctl status nginx 2>/dev/null | head -3 || echo "（nginx 未安装，仅演示排查逻辑）"
echo ""
! echo "=== 场景 2：磁盘空间消失 ==="
! echo "模拟：创建 30MB 文件 -> rm 删除但保留 fd"
! exec 4> /tmp/.big_hidden 2>/dev/null
! dd if=/dev/zero of=/tmp/.big_hidden bs=1M count=30 2>/dev/null
! rm -f /tmp/.big_hidden
! echo "文件已删除。查找 deleted 文件："
! lsof 2>/dev/null | grep deleted | head -3 || echo "（lsof 受限）"
! exec 4>&-
! echo "fd 4 已关闭"
echo ""
! echo "=== 场景 3：OOM Killer ==="
! dmesg 2>/dev/null | grep -i "killed process" | tail -5 || echo "没有 OOM 记录（系统运行正常）"
! echo ""
! echo "OOM 排查命令："
! echo "  dmesg | grep -i oom"
! echo "  dmesg | grep -i 'killed process'"
! echo "  cat /proc/<pid>/oom_score"
! echo "  echo -1000 > /proc/<pid>/oom_score_adj"
echo ""
! echo "=== 场景 4：网络超时 ==="
! echo "DNS 解析测试："
! if command -v dig &>/dev/null; then
!     dig +short example.com 2>/dev/null || echo "DNS 解析失败"
! else
!     host example.com 2>/dev/null || echo "dig/host 不可用"
! fi
! echo "TCP 连接状态："
! ss -tan | head -5
echo ""
! echo "=== 场景 5：CPU 100% ==="
! echo "当前 CPU 最高的进程："
! ps aux --sort=-%cpu 2>/dev/null | awk 'NR<=3 {printf "  PID=%-6s %-25s CPU=%s%%\n", $2, $11, $3}' || echo "无权限"
echo ""
! echo "=== 场景 6：容器内慢 ==="
! if command -v docker &>/dev/null; then
!     docker stats --no-stream 2>/dev/null | head -5 || echo "Docker 未运行"
! fi
! echo "容器慢常见原因："
! echo "  1. cgroup 资源限制太低（--cpus / --memory）"
! echo "  2. OverlayFS 文件系统 I/O 开销"
! echo "  3. 存储驱动选择（overlay2 vs devicemapper）"
! echo "  4. 网络经过额外 bridge 层"
echo ""
! echo "=== 排障笔记模板 ==="
! echo "对每个场景记录：现象、排查过程、根因、修复方案、预防措施"

---
## 第4周总结

### 核心概念

| 概念 | 一句话 |
|------|--------|
| **load average** | 进程队列长度，超过 CPU 核心数说明有瓶颈 |
| **top us/sy/wa** | 用户代码/内核代码/I/O 等待的 CPU 时间 |
| **perf + 火焰图** | 函数级 CPU 采样，宽度 = CPU 时间占比 |
| **VIRT/RES/SHR** | 虚拟/物理/共享内存，排内存看 RES |
| **OOM Killer** | 内存不足时选进程杀掉，查 dmesg |
| **iostat %util** | 磁盘忙碌度 > 80% 说明瓶颈 |
| **strace** | 追踪系统调用，排查启动失败/权限/配置文件 |
| **lsof | grep deleted** | 找隐藏大文件（经典问题） |
| **cgroup v2** | CPU/内存/I/O/进程数限制内核机制 |
| **故障排查方法论** | 现象 -> 工具链 -> 根因 -> 修复 -> 预防 |

### 本周命令速查

```bash
top / htop                         # 实时进程监控
mpstat -P ALL 1                    # 每个 CPU 核心
pidstat -p <PID> 1                 # 按进程统计
perf top -p <PID>                  # 函数级热点
perf record -p <PID> -g sleep 30  # 采样
free -h                            # 内存使用
dmesg | grep -i oom                # OOM 日志
cat /proc/<pid>/smaps              # 进程内存映射
pmap -x <pid>                      # 内存映射快照
iostat -x 1                        # 磁盘 I/O
fio --name=test --rw=randrw        # 磁盘基准测试
strace -p <pid>                    # 追踪系统调用
strace -e trace=file <cmd>         # 只追踪文件
lsof -i :80                        # 端口占用
lsof | grep deleted                # 隐藏大文件
cat /sys/fs/cgroup/cpu.max         # cgroup CPU 限制
echo val > /sys/fs/cgroup/xxx/memory.max  # cgroup 内存
```

### 第四周里程碑

- 能独立排查 CPU/内存/磁盘/网络性能问题
- 会用 strace 调试系统调用级问题
- 会用 lsof 排查一切文件相关问题
- 理解 cgroup v2 资源限制原理
- 具备系统化的故障排查方法论

### 推荐资源

| 类型 | 资源 |
|------|------|
| 书籍 | 《性能之巅》(Systems Performance, Brendan Gregg) |
| 网站 | brendangregg.com |
| 实战 | sadservers.com -- Linux 故障排除挑战 |
| 工具 | tldr（npm install -g tldr）比 man 更易懂 |

---
**恭喜完成全部四周的 Linux 深入学习！**

现在你可以：
- 深入理解文件系统、进程、Shell 编程
- 管理系统服务、用户权限、存储和包管理
- 理解 TCP/IP 和容器网络原理
- 独立排查系统级性能和故障问题
